# Testing environment for training a Neural Network for Localization


In [1]:
import os

import numpy as np

from scripts.data_loader import load_dataframe

# source files
BASE_DIR = "data/"
FULL_DATA_SET = "Campaign_data_NBIoT_1_2_3_4_5_6_interpolated_smoothed.mat"
filename = os.path.join(BASE_DIR, FULL_DATA_SET)

# Series of random seeds for reproducability
random_seeds = np.loadtxt('data/random_seeds.csv', dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename)

Loaded dataframe from .h5 file: data/Campaign_data_NBIoT_1_2_3_4_5_6_interpolated_smoothed.mat_dataframe.h5


## Prepare the data for model training

In [2]:
from scripts.weighted_coverage import create_point_matrix
from scripts.utils import extract_unique_npcis, haversine_distance
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from tensorflow import keras
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import LearningRateScheduler
from tensorflow.keras import Input
from pyproj import CRS, Transformer

wgs84 = CRS.from_epsg(4326)
utm = CRS.from_epsg(32633)  # Example UTM zone 33N


def transform_to_utm(coords: np.array) -> np.array:
    wgs84 = CRS.from_epsg(4326)
    utm = CRS.from_epsg(32633)  # Example UTM zone 33N
    transformer_to_utm = Transformer.from_crs(wgs84, utm, always_xy=True)
    return np.array([transformer_to_utm.transform(lng, lat) for lat, lng in coords])


def transform_to_wsg84(coords: np.array) -> np.array:
    wgs84 = CRS.from_epsg(4326)
    utm = CRS.from_epsg(32633)  # Example UTM zone 33N
    transformer_to_wsg84 = Transformer.from_crs(utm, wgs84, always_xy=True)
    return np.array([transformer_to_wsg84.transform(lat, lng) for lat, lng in coords])


def prepare_data(df: pd.DataFrame, unique_npcis: np.array, rf_param, test_size: float = 0.3, random_seed: int = 42):
    """
    Prepares the dataset for model training and testing
    Given a random seed for reproducability
    
    :param df: 
    :param unique_npcis: 
    :param rf_param: 
    :param test_size: 
    :param random_seed: 
    :return: 
    """
    # Shuffle the dataframe
    df = df.sample(frac=1, random_state=random_seed).reset_index(drop=True)

    # Get the point matrix (row of NSINR values for each unique <NPCI, eNodeB-ID, operatorID> triplet
    point_matrix, _ = create_point_matrix(df, unique_npcis, rf_param)

    X = point_matrix
    y = df[['lat', 'lng']].values

    # Transform coordinates to UTM
    y_utm = transform_to_utm(y)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y_utm, test_size=test_size, random_state=random_seed)

    # Min-Max normalization
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()

    X_train = scaler_X.fit_transform(X_train)
    X_test = scaler_X.transform(X_test)
    y_train = scaler_y.fit_transform(y_train)
    y_test = scaler_y.transform(y_test)

    return X_train, X_test, y_train, y_test, scaler_X, scaler_y


def train_model(X_train, y_train):
    """
    Trains the Neural Network
    :param X_train: 
    :param y_train: 
    :return: 
    """

    # Define the model
    def build_model(input_shape):
        model = keras.Sequential([
            Input(shape=input_shape, name='input_layer'),
            Dense(600, activation='relu', name='dense_600'),
            BatchNormalization(name='batch_norm_600'),
            Dense(300, activation='relu', name='dense_300'),
            BatchNormalization(name='batch_norm_300'),
            Dense(150, activation='relu', name='dense_150'),
            BatchNormalization(name='batch_norm_150'),
            Dropout(0.2, name='dropout'),
            Dense(2, name='output_layer')
        ])
        return model

    # Example usage
    input_shape = (X_train.shape[1],)
    model = build_model(input_shape)

    # Compile the model
    initial_lr = 0.1
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=initial_lr),
                  loss='mean_squared_error',
                  metrics=['mae'])

    # Learning rate scheduler
    def scheduler(epoch, lr):
        if epoch % 20 == 0 and epoch:
            return lr * 0.1
        return lr

    lr_scheduler = LearningRateScheduler(scheduler)

    # Train the model
    model.fit(X_train, y_train, epochs=60, batch_size=48, validation_split=0.2, callbacks=[lr_scheduler], verbose=0)

    return model


def model_predict(model, X_test, scaler_X, scaler_y):
    """
    Use the model to predict location of TPs
    return the predicted positions in the WSG84 format
    
    :param model: 
    :param X_test: 
    :param scaler_X: 
    :param scaler_y: 
    :return: 
    """
    y_pred = model.predict(X_test, verbose=0)
    y_pred = scaler_y.inverse_transform(y_pred)
    y_pred_wgs84 = transform_to_wsg84(y_pred)

    return y_pred_wgs84




In [12]:
from scripts.utils import RF_PARAM

# Experiement setup for the ANN
rf_param = RF_PARAM.NSINR
operator_choice = np.array([1, 10, 88])
unique_npcis = extract_unique_npcis(df, operator_choice)
n_runs = 40
results = []

print(f"""
Building, training and testing Neural Network

🧪 Experiment setup 🧪 
⚙️ RF PARAM {rf_param}
📶 Operator choice {operator_choice}
🔁 Number of runs {n_runs}
_________________________________
""")

for i in range(n_runs):
    print(f"\r🔄 {i + 1} / {n_runs}", end="")

    # Prepare the data
    X_train, X_test, y_train, y_test, scaler_X, scaler_y = prepare_data(df, unique_npcis, rf_param,
                                                                        random_seed=random_seeds[i])
    # Time and train the model 
    start = time.time()
    model = train_model(X_train, y_train)
    offline_runtime = time.time() - start

    # Evaluate the model
    test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)

    # Time the prediction and transformation time
    start = time.time()
    predicted = model_predict(model, X_test, scaler_X, scaler_y)
    online_runtime = time.time() - start

    # Convert test labels back to WGS84 for comparison
    #y_test_wgs84 = np.array([transformer_to_wgs84.transform(x, y) for x, y in scaler_y.inverse_transform(y_test)])
    y_test_wgs84 = transform_to_wsg84(scaler_y.inverse_transform(y_test))
    # Calculate error
    errors = haversine_distance(y_test_wgs84[:, 0], y_test_wgs84[:, 1], predicted[:, 0], predicted[:, 1])

    results.append(
        (i,
         offline_runtime,
         online_runtime,
         test_loss,
         test_mae,
         errors.mean(),
         errors.std(),
         errors.min(),
         errors.max()
         )
    )

    print(f"\r✅ {i + 1} run completed")

result_df = pd.DataFrame(results,
                         columns=['Run', 'offline_runtime', 'online_runtime', 'test_loss', 'test_mae', 'mean_pos_error',
                                  'std_pos_error', 'min_pos_error', 'max_pos_error'])

mean_error = result_df['mean_pos_error'].mean()

print(f"""_________________________________
📊 Result over {n_runs} runs
📏 Mean error: {mean_error:.2f}
""")


Building, training and testing Neural Network

🧪 Experiment setup 🧪 
⚙️ RF PARAM RF_PARAM.NSINR
📶 Operator choice [ 1 10 88]
🔁 Number of runs 40
_________________________________

✅ 1 run completed
✅ 2 run completed
✅ 3 run completed
✅ 4 run completed
✅ 5 run completed
✅ 6 run completed
✅ 7 run completed
✅ 8 run completed
✅ 9 run completed
✅ 10 run completed
✅ 11 run completed
✅ 12 run completed
✅ 13 run completed
✅ 14 run completed
✅ 15 run completed
✅ 16 run completed
✅ 17 run completed
✅ 18 run completed
✅ 19 run completed
✅ 20 run completed
✅ 21 run completed
✅ 22 run completed
✅ 23 run completed
✅ 24 run completed
✅ 25 run completed
✅ 26 run completed
✅ 27 run completed
✅ 28 run completed
✅ 29 run completed
✅ 30 run completed
✅ 31 run completed
✅ 32 run completed
✅ 33 run completed
✅ 34 run completed
✅ 35 run completed
✅ 36 run completed
✅ 37 run completed
✅ 38 run completed
✅ 39 run completed
✅ 40 run completed
_________________________________
📊 Result over 40 runs
📏 Mean error